# 相対比較（複数選択）で創薬標的の妥当性を検証する

ノートブック 03 は遺伝子1つずつに「Yes/No」で答えさせる**絶対評価**でした。絶対評価には次の弱点があります。
- 「Yesと言いやすい／言いにくい」というモデルの癖（尤度の偏り）が、そのままスコアに乗ってしまう
- 質問文の言い回し1つで、known遺伝子の一部だけ極端に低くなることがある（例：ノートブック03のB2・B4）

このノートブックでは代わりに、**相対評価（強制選択）**を試します。

1. 複数の遺伝子をランダムにグループ化して並べ、「この中で最も関連性が高いのはどれか」を**番号で**答えさせる（遺伝子名そのものを答えさせない。綴りの揺れ・スペル一致の問題を避けるため）
2. 同じ遺伝子を何度も（グループの組み合わせを変えて）比較させ、選ばれた確率の平均を「関連性スコア」とする
3. 「この中にはっきり関連する遺伝子はない」という**その他（None）**の選択肢も用意する
4. グループの人数（デフォルト5人＋比較用に他のサイズ）と、質問の言い回し（3パターン）を変えて、どの設計がknown（既知標的）とrandom（無関係な遺伝子）を最もよく見分けられるかをAUCで比較する

絶対評価（ノートブック03）と相対評価（このノートブック）の**両方**でknownが高くrandomが低くなるなら、その判定はより信頼できます。逆に一方でしか効かない質問は、その聞き方固有の癖を測っているだけの可能性があります。

## 前提
- ノートブック03と同じ `data/diseases.json` の疾患・`data/genes/*.tsv` の遺伝子リストを使います。別ファイル（.pyファイル）は参照せず、コードはこのノートブック内で完結させています。
- 本命はノートブック03と同じく、GGUFをllama-cpp-pythonで直接動かす経路です（前置きのKVキャッシュ保存・復元が使えます）。
- モデルが無い環境ではモック（擬似乱数）で動きます。数値に意味はありません。
- 選択肢の番号は必ず1桁（1〜9）に収まるようにしてください（GROUP_SIZE + Noneの分 ≤ 9）。2桁になるとトークンが「1」「0」のように割れて確率の読み取りが複雑になるためです。

### このセルがすること：準備

In [ ]:
import os, re, math, glob, json, time, random, hashlib, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
ROOT = os.path.abspath("..")
print("ready:", ROOT)

## 疾患の選択

### このセルがすること：`data/diseases.json` の登録疾患から1つ選び、病名・症状の箇条書き・遺伝子リストを読み込む
- ノートブック03と全く同じ読み込み方です。`DISEASE_KEY` を変えるだけで切り替わります：`ra` / `scz` / `cystinuria` / `prostate_cancer` / `achondroplasia`。

In [ ]:
DISEASE_KEY = "ra"                    # "ra" / "scz" / "cystinuria" / "prostate_cancer" / "achondroplasia"
GENE_SET = "set100"                   # "set100" / "set1000" / "known" / "candidates"

REGISTRY = json.load(open(os.path.join(ROOT, "data", "diseases.json"), encoding="utf-8"))
print("登録疾患:", {k: v["name"] for k, v in REGISTRY.items()})
D = REGISTRY[DISEASE_KEY]
DISEASE = D["name"]
DISEASE_INFO = D["info"][:5]
GENE_FILE = os.path.join(ROOT, "data", "genes", f"{D['gene_prefix']}_{GENE_SET}.tsv")
print("選択:", DISEASE, "| 遺伝子リスト:", os.path.basename(GENE_FILE))
for b in DISEASE_INFO: print("  -", b[:110] + ("…" if len(b) > 110 else ""))

## 設定

### このセルがすること：グループの人数・ラウンド数・「その他」選択肢の有無・モデルを決める
- `GROUP_SIZES`：1回の質問に並べる遺伝子の数。デフォルト5に加え、比較用に別の人数もリストで指定できます。
- `INCLUDE_NONE`：「この中にはっきり関連する遺伝子はない」という選択肢を追加するか。
- `N_ROUNDS`：全遺伝子をシャッフルして重複なくグループ分けする回数。1回のラウンドで全遺伝子が1回ずつ、別々の相手と比較されます。`N_ROUNDS` を増やすほど、1遺伝子あたりの比較回数が増えて推定が安定します。
- モデルの選び方はノートブック03と同じ仕組み（`~/llm/models` のGGUF、またはOllama）です。

In [ ]:
MAX_GENES = None                                   # 試運転なら 20 など

GROUP_SIZES = [5, 8]                               # 1回の質問に並べる遺伝子の数（デフォルト5 ＋ 比較用に8）
INCLUDE_NONE = True                                # 「この中にはっきり関連する遺伝子はない」という選択肢を追加する
N_ROUNDS = 20                                      # 全遺伝子を1回ずつ含む「ラウンド」を何回繰り返すか（≒1遺伝子あたりの比較回数）
STRICT_PROMPT = True
N_CTX = 2048

for g in GROUP_SIZES:                              # 選択肢は必ず1桁（1〜9）に収める
    n_opt = g + (1 if INCLUDE_NONE else 0)
    assert n_opt <= 9, f"group_size={g} は選択肢が{n_opt}個になり2桁の番号が必要です。9以下にしてください。"

# --- モデルの選択（ノートブック03と同じ仕組み） ---
BACKEND = "auto"                                   # "auto" / "gguf" / "ollama" / "mock"
MODEL_DIR = os.path.expanduser("~/llm/models")
OLLAMA_URL = "http://localhost:11434"
MODEL_SELECT = "auto"                              # "auto" / 一覧の番号 / 名前の一部
PREFER = ("txgemma", "medgemma", "gemma")
models = []
for h in sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True)):
    models.append({"kind": "gguf", "name": os.path.basename(h), "path": h, "size_gb": round(os.path.getsize(h) / 1e9, 2)})
try:
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
        for m in json.load(r).get("models", []):
            models.append({"kind": "ollama", "name": m["name"], "path": m["name"], "size_gb": round(m.get("size", 0) / 1e9, 2)})
except Exception:
    pass
chosen = None
if BACKEND != "mock":
    if isinstance(MODEL_SELECT, int):
        chosen = models[MODEL_SELECT]
    else:
        pool = [m for m in models if BACKEND == "auto" or m["kind"] == BACKEND]
        if MODEL_SELECT != "auto": pool = [m for m in pool if MODEL_SELECT.lower() in m["name"].lower()]
        ranked = sorted(pool, key=lambda m: (min([i for i, p in enumerate(PREFER) if p in m["name"].lower()] or [99]),
                                             0 if m["kind"] == "gguf" else 1, m["name"]))
        chosen = ranked[0] if ranked else None
USE_LLM = chosen is not None
BACKEND_USED = chosen["kind"] if chosen else "mock"
MODEL_PATH = chosen["path"] if chosen else None
print("使えるモデル:"); [print(f"  [{i}] {m['kind']:6s} {m['size_gb']:6.2f} GB  {m['name']}") for i, m in enumerate(models)]
print("選択:", f"{BACKEND_USED}: {MODEL_PATH}" if USE_LLM else "モック（擬似乱数。数値に意味なし）")
OUT_DIR = os.path.join(ROOT, "outputs"); os.makedirs(OUT_DIR, exist_ok=True)

## 入力遺伝子

### このセルがすること：遺伝子リストを読み、プロンプトに入れる名前（記号＋タンパク質名）を作る
- ノートブック03と同じ形式です。タンパク質名は `protein_name_uniprot` → `gene_name` → 記号のみ、の順で使います。

In [ ]:
genes = pd.read_csv(GENE_FILE, sep="\t", dtype=str).fillna("")
for col in ("protein_name_uniprot", "gene_name", "category", "label"):
    if col not in genes.columns: genes[col] = ""
if MAX_GENES: genes = genes.head(MAX_GENES).copy()
genes["gene_label"] = [f"{g['symbol']} ({(g['protein_name_uniprot'] or g['gene_name']).split('|')[0]})" if (g["protein_name_uniprot"] or g["gene_name"]) else g["symbol"] for _, g in genes.iterrows()]
print(len(genes), "genes"); genes[["symbol", "gene_label", "category"]].head(5)

## 質問パターンとプロンプトの組み立て

### このセルがすること：3種類の質問パターンと、共通の前置き・番号付き遺伝子リストの組み立て方を定義する
- `QUESTION_VARIANTS`：3つの角度から「最も関連性が高い遺伝子はどれか」を聞きます。`target`（あらゆる証拠を総合した判断）、`mechanism`（自分自身の基質・経路・細胞・回路が説明文と一致するか、ノートブック03のB1に相当）、`evidence`（既存薬・ヒト遺伝学の直接証拠、ノートブック03のA1/A2に相当）。
- 前置き（`prefix_text`）は「役割・注意書き・病名・症状・答え方の指示」で、**グループの人数ごとに1文字も変わらない**ので、人数ごとに1回だけ処理して保存できます（ノートブック03と同じKVキャッシュの考え方）。
- `group_block`：遺伝子ラベルのリストを「1. 記号 (タンパク質名)」の番号付きリストに変換します。`INCLUDE_NONE=True` なら、最後に「該当なし」の番号を追加します。
- `question_line`：質問パターンの文言を病名で埋め、末尾に「Answer:」を付けます。

In [ ]:
QUESTION_VARIANTS = {
    "target":    "Which numbered gene above is the single most plausible drug target for {disease}? Weigh all forms of evidence together "
                 "(existing drugs, human genetics, and mechanism).",
    "mechanism": "Which numbered gene above plays the most central role in causing {disease}, based on its OWN specific substrate, "
                 "signalling pathway, cell type or circuit — as opposed to a related but functionally or anatomically distinct one?",
    "evidence":  "Which numbered gene above has the strongest existing evidence for {disease} — an approved drug target, or human genetic "
                 "variants that cause {disease} or alter its risk or severity?",
}
VIDS = list(QUESTION_VARIANTS.keys())

STRICT_NOTE = ("Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, "
               "specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.\n")

def prefix_text(group_size):
    """全遺伝子で共通の前置き（グループの人数ごとに1回だけ処理して保存する）。"""
    info = "\n".join(f"- {b}" for b in DISEASE_INFO)
    n_opt = group_size + (1 if INCLUDE_NONE else 0)
    return ("You are an expert in drug discovery and human disease biology. You will be shown a numbered list of "
            f"{group_size} candidate genes for one disease, and asked which ONE number is the best answer to a question.\n"
            + (STRICT_NOTE if STRICT_PROMPT else "") +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{info}\n"
            f"Answer with a single number from 1 to {n_opt} only. No words, no explanation.\n\n")

def group_block(gene_labels):
    """遺伝子ラベルのリストを『1. 記号 (タンパク質名)』の番号付きリストにする。INCLUDE_NONE なら最後に『該当なし』を足す。"""
    lines = [f"{i + 1}. {g}" for i, g in enumerate(gene_labels)]
    if INCLUDE_NONE:
        lines.append(f"{len(gene_labels) + 1}. None of the above genes seem clearly relevant")
    return "Candidate genes:\n" + "\n".join(lines) + "\n"

def question_line(vid):
    text = QUESTION_VARIANTS[vid].format(disease=DISEASE)
    return f"Question ({vid}): {text} Answer:"

print(prefix_text(5))
print(group_block(["TNF (tumor necrosis factor)", "PADI4 (peptidyl arginine deiminase 4)", "IL6 (interleukin 6)",
                    "BRCA1 (breast cancer 1)", "ACTB (actin beta)"]))
print(question_line("target"))

## エンジン（GGUF を llama-cpp-python で直接動かす）

### `last_logprobs()` / `option_logprob(lp, ids)` — ノートブック03と同じ、綴り違いを合算した対数確率の読み取り
どんな def か：直前に処理した位置の全語彙の対数確率を読み、指定したトークンid群を綴り違いとして合算（log-sum-exp）します。
return：`last_logprobs` は numpy配列（語彙数）、`option_logprob` は float。

### `normalize_option_probs(logps)` — 選択肢だけでソフトマックス正規化する
どんな def か：複数の選択肢（例：番号1〜6）の対数確率の辞書を受け取り、その選択肢の中だけで合計1になる確率に変換します（他の語彙は無視する）。
return：dict（選択肢番号 → 確率）。

### `score_group_gguf(group_size, gene_labels)` — 1グループぶん、3つの質問パターン全部を1回の処理で採点する
各ステップ：
1. 保存しておいた、そのグループ人数の前置き状態を `load_state` で復元する（前置きは再処理しない）。
2. 番号付きの遺伝子リストを処理する。
3. 質問パターンごとに、質問文を処理し「Answer:」の位置で各選択肢番号の確率を読み、選択肢だけでソフトマックス正規化する。
4. モデル自身の答え（最も確率が高い番号）のトークンと改行を処理して、次の質問パターンへ進む（ノートブック03の6問チェーンと同じ条件付けの考え方）。
return：`{質問パターンid: {選択肢番号: 確率}}` の辞書。

### このセルがすること：エンジンを読み込み、グループの人数ごとに前置きを処理して状態を保存し、上のdefを定義する
- Ollamaでは前置きの再処理を毎回行うため遅い経路になります（ノートブック03と同じ二段階のlogprobs対応確認＋非対応時のサンプリング代用付き）。

In [ ]:
DIGIT_SPELL = {n: [str(n), f" {n}"] for n in range(1, 10)}   # 数字1桁の綴り違い（例："5" と " 5"）
prefix_state = {}    # group_size -> (state, n_tokens)  GGUFのみ

if BACKEND_USED == "gguf":
    from llama_cpp import Llama
    llm = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, logits_all=False, verbose=False)
    def tok(text, bos=False): return llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    def ids_of(spellings):
        out = []
        for s in spellings:
            t = tok(s)
            if len(t) == 1 and t[0] not in out: out.append(t[0])
        return out
    NL = tok("\n")
    for g in GROUP_SIZES:                                          # グループの人数ごとに前置きを1回だけ処理して保存
        llm.reset(); llm.eval(tok(prefix_text(g), bos=True))
        prefix_state[g] = (llm.save_state(), llm.n_tokens)
    print(f"GGUF engine ready | group sizes: {GROUP_SIZES} | prefix tokens: {[prefix_state[g][1] for g in GROUP_SIZES]}")
elif BACKEND_USED == "ollama":
    print("Ollama を使います（前置きのキャッシュ保存はできないため、質問ごとに1回ずつ呼ぶ遅い経路です）:", MODEL_PATH)
else:
    print("警告: モデルが無いのでモック（擬似乱数）です。")

def last_logprobs():
    lg = np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
    lg = lg - lg.max()
    return lg - math.log(np.exp(lg).sum())

def option_logprob(lp, ids):
    v = [lp[i] for i in ids]; m = max(v)
    return m + math.log(sum(math.exp(x - m) for x in v))

def normalize_option_probs(logps):
    m = max(logps.values())
    exps = {k: math.exp(v - m) for k, v in logps.items()}
    z = sum(exps.values())
    return {k: v / z for k, v in exps.items()}

def score_group_gguf(group_size, gene_labels):
    st, n = prefix_state[group_size]
    llm.reset()                                                      # 1a. 前回までの状態を完全に空にする（長さの違う状態を上書きすると
                                                                       #     KVキャッシュが不整合になり llama_decode が失敗するため）
    llm.load_state(st)                                               # 1b. 前置きの状態を復元
    llm.eval(tok(group_block(gene_labels)))                          # 2. 番号付き遺伝子リスト
    n_opt = group_size + (1 if INCLUDE_NONE else 0)
    OPT_IDS = {i: ids_of(DIGIT_SPELL[i]) for i in range(1, n_opt + 1)}
    out = {}
    for vid in VIDS:
        llm.eval(tok(question_line(vid)))                           # 3. 質問文 → 「Answer:」の位置
        lp = last_logprobs()
        logps = {i: option_logprob(lp, OPT_IDS[i]) for i in range(1, n_opt + 1)}
        probs = normalize_option_probs(logps)
        out[vid] = probs
        best = max(probs, key=probs.get)
        llm.eval(ids_of(DIGIT_SPELL[best])[:1] + NL)                 # 4. モデル自身の答えで条件付け
    return out

def sequence_logprob(prefix, continuation):
    llm.reset(); llm.eval(tok(prefix, bos=True))
    total = 0.0
    for t in tok(continuation):
        total += float(last_logprobs()[t]); llm.eval([t])
    return total

# ---- Ollama（遅い経路）: 質問ごとに raw テンプレートで1回ずつ ----
def ollama_template(prompt_body):
    n = str(MODEL_PATH).lower()
    if "gemma" in n:        tpl = "<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\nAnswer:"
    elif "qwen3" in n:      tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nAnswer:"
    elif "qwen" in n or "deepseek" in n: tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\nAnswer:"
    elif "mistral" in n:    tpl = "[INST] {body} [/INST] Answer:"
    elif "llama" in n:      tpl = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{body}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAnswer:"
    else:                   tpl = "{body}\nAnswer:"
    return tpl.format(body=prompt_body)

OLLAMA_LP_ENDPOINT = None

def ollama_post(path, body):
    req = urllib.request.Request(OLLAMA_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

def ollama_top_logprobs(full_prompt, k=20):
    global OLLAMA_LP_ENDPOINT
    tries = [OLLAMA_LP_ENDPOINT] if OLLAMA_LP_ENDPOINT else ["/api/generate", "/v1/completions"]
    for ep in tries:
        try:
            if ep == "/api/generate":
                out = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "raw": True, "stream": False,
                                       "logprobs": True, "top_logprobs": k, "options": {"temperature": 0, "num_predict": 1}})
                lps = out.get("logprobs") or []
                top = {t["token"]: t["logprob"] for t in (lps[0].get("top_logprobs", []) if lps else [])}
                if lps and not top: top = {lps[0]["token"]: lps[0]["logprob"]}
            else:
                ch = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "max_tokens": 1, "temperature": 0,
                                      "logprobs": k})["choices"][0]
                lp = ch.get("logprobs") or {}
                if lp.get("content"): top = {t["token"]: t["logprob"] for t in lp["content"][0].get("top_logprobs", [])}
                elif lp.get("top_logprobs"): top = dict(lp["top_logprobs"][0])
                else: top = {}
            if top:
                OLLAMA_LP_ENDPOINT = ep
                return top
        except Exception:
            continue
    return None

def ollama_option_probs(prompt_body, n_opt):
    top = ollama_top_logprobs(ollama_template(prompt_body), 20)
    if top is None:
        return None
    def lse(vals): m = max(vals); return m + math.log(sum(math.exp(v - m) for v in vals))
    floor = min(top.values()) if top else -20
    logps = {}
    for i in range(1, n_opt + 1):
        vals = [v for t, v in top.items() if t.strip() == str(i)]
        logps[i] = lse(vals) if vals else floor
    return normalize_option_probs(logps)

def ollama_generate_number(prompt_body, n_opt, temperature=1.0):
    out = ollama_post("/api/generate", {"model": MODEL_PATH, "prompt": ollama_template(prompt_body), "raw": True,
                                        "stream": False, "options": {"temperature": temperature, "num_predict": 3}})
    m = re.match(r"\s*(\d+)", out.get("response") or "")
    v = int(m.group(1)) if m else None
    return v if v is not None and 1 <= v <= n_opt else None

def score_group_ollama(group_size, gene_labels):
    n_opt = group_size + (1 if INCLUDE_NONE else 0)
    out = {}
    for vid in VIDS:
        body = prefix_text(group_size) + group_block(gene_labels) + question_line(vid).replace(" Answer:", "")
        probs = ollama_option_probs(body, n_opt)
        if probs is None:                                     # logprobs 非対応 → 8回サンプリングで代用
            counts = {i: 0 for i in range(1, n_opt + 1)}
            for _ in range(8):
                v = ollama_generate_number(body, n_opt)
                if v is not None: counts[v] += 1
            total = sum(counts.values()) or 1
            probs = {i: (c + 0.5) / (total + 0.5 * n_opt) for i, c in counts.items()}
        out[vid] = probs
    return out

def score_group_mock(group_size, gene_labels):
    n_opt = group_size + (1 if INCLUDE_NONE else 0)
    out = {}
    for vid in VIDS:
        h = [int(hashlib.md5((g + vid + str(i)).encode()).hexdigest(), 16) % 1000
             for i, g in enumerate(gene_labels + (["NONE"] if INCLUDE_NONE else []))]
        z = sum(h) or 1
        out[vid] = {i + 1: h[i] / z for i in range(n_opt)}
    return out

score_group = {"gguf": score_group_gguf, "ollama": score_group_ollama}.get(BACKEND_USED, score_group_mock)

if BACKEND_USED == "ollama":
    probe = ollama_top_logprobs(ollama_template("Pick 1 or 2.\n1. cats\n2. dogs\nAnswer:"), 5)
    print(f"logprobs: 対応（{OLLAMA_LP_ENDPOINT}） 例 {dict(list(probe.items())[:3])}" if probe is not None else
          "logprobs: 非対応 → 8回サンプリングで代用します")

demo_labels = genes["gene_label"].head(GROUP_SIZES[0]).tolist()
t0 = time.time(); demo = score_group(GROUP_SIZES[0], demo_labels); dt = time.time() - t0
print("test group:", demo_labels)
for vid, probs in demo.items(): print(" ", vid, {k: round(v, 3) for k, v in probs.items()})
print(f"{dt:.2f}s for {len(VIDS)} variants")

## ラウンド設計（ランダムに重複なく分割するのを繰り返す）と実行

### このセルがすること：全遺伝子をシャッフルして重複なくグループ分けする「ラウンド」を `N_ROUNDS` 回繰り返し、各遺伝子の選ばれやすさを集計する
- **なぜ毎回シャッフルするのか**：もし同じ遺伝子が毎回同じ番号（例えば常に1番）に置かれると、モデルの「若い番号を選びやすい」という癖（位置バイアス）がそのまま結果に乗ってしまいます。ラウンドごとに全体をシャッフルしてグループと番号をランダムに決め直すことで、この癖を統計的に打ち消します（実際に効いているかは、下の評価セルで位置ごとの平均確率を見て確認します）。
- 1ラウンドでは、シャッフルした全遺伝子を `GROUP_SIZES` の人数ごとに先頭から順に切り出してグループを作ります（重複なく全遺伝子が1回ずつ登場）。人数で割り切れない端数グループは、他の遺伝子から補充します（ごく少数なので誤差は無視できます）。
- 1グループにつき、3つの質問パターン（`VIDS`）すべてを1回の処理で聞きます。
- 各遺伝子について、`sum_p / n`（選ばれた確率の平均）と `wins / n`（実際に一番確率が高かった回数の割合）を、質問パターンごとに集計します。

In [ ]:
rows = []
diag_positions = []   # 位置バイアスの確認用: (group_size, position, prob)
none_diag = []          # 「その他」診断用: (group_size, variant, prob_none, is_none_argmax)
t0 = time.time()
rng = random.Random(0)
for gsize in GROUP_SIZES:
    idx = list(genes.index)
    stats = {i: {vid: {"sum_p": 0.0, "n": 0, "wins": 0} for vid in VIDS} for i in idx}
    n_groups_done = 0
    for rnd in range(N_ROUNDS):
        order = idx[:]; rng.shuffle(order)
        for start in range(0, len(order), gsize):
            chunk = order[start:start + gsize]
            if len(chunk) < gsize:                      # 端数は他の遺伝子から補充する（重複可、稀なので無視できる誤差）
                chunk = chunk + rng.sample(idx, gsize - len(chunk))
            labels = [genes.loc[i, "gene_label"] for i in chunk]
            probs_by_variant = score_group(gsize, labels)
            n_groups_done += 1
            for vid, probs in probs_by_variant.items():
                best = max(probs, key=probs.get)
                if INCLUDE_NONE:
                    none_diag.append((gsize, vid, probs.get(gsize + 1, 0.0), best == gsize + 1))
                for slot, gidx in enumerate(chunk):
                    p = probs.get(slot + 1, 0.0)
                    stats[gidx][vid]["sum_p"] += p
                    stats[gidx][vid]["n"] += 1
                    if best == slot + 1: stats[gidx][vid]["wins"] += 1
                    diag_positions.append((gsize, slot + 1, p))
    for i in idx:
        g = genes.loc[i]
        for vid in VIDS:
            s = stats[i][vid]
            rows.append({"group_size": gsize, "variant": vid, "symbol": g["symbol"], "gene_label": g["gene_label"],
                        "category": g["category"], "label": g["label"],
                        "mean_p": round(s["sum_p"] / s["n"], 4) if s["n"] else float("nan"),
                        "win_rate": round(s["wins"] / s["n"], 4) if s["n"] else float("nan"), "n_trials": s["n"]})
    print(f"group_size={gsize}: {n_groups_done} グループ × {len(VIDS)}質問 処理 ({time.time()-t0:.0f}s)")
res = pd.DataFrame(rows)
model_tag = f"{BACKEND_USED}:{os.path.basename(str(MODEL_PATH))}" if USE_LLM else "MOCK"
res["model"] = model_tag; res["disease"] = DISEASE
OUT_CSV = os.path.join(OUT_DIR, os.path.basename(GENE_FILE).replace(".tsv", "") + "_rank.csv")
res.to_csv(OUT_CSV, index=False)
print(f"elapsed {time.time()-t0:.0f}s | wrote {OUT_CSV}")

## 評価

### このセルがすること：group_size × 質問パターンの組み合わせごとにAUCを比較し、位置バイアスと「その他」の使われ方を診断する
- `mean_p`（選ばれた確率の平均）と `win_rate`（一番確率が高かった回数の割合）の両方で、known vs random のAUCを計算します。
- **位置バイアスの診断**：もし特定の番号（例えば1番）が内容に関係なく選ばれやすいなら、番号ごとの平均確率が均等（およそ `1/選択肢数`）からずれます。
- **「その他」の診断**：ランダム遺伝子ばかりのグループで「その他」がよく選ばれ、known遺伝子が混ざると選ばれにくくなっていれば、この選択肢が意図通り機能している証拠です。

In [ ]:
def auc(pos, neg):
    pos, neg = list(pos), list(neg)
    if not pos or not neg: return float("nan")
    return sum(1.0 if a > b else 0.5 if a == b else 0.0 for a in pos for b in neg) / (len(pos) * len(neg))

if not USE_LLM: print("警告: モックの数値です。")

known, other, rnd_cat, cand = (res["category"] == "known"), (res["category"] != "known"), (res["category"] == "random"), (res["category"] == "candidate")
combo_rows = []
for gsize in GROUP_SIZES:
    for vid in VIDS:
        m = (res["group_size"] == gsize) & (res["variant"] == vid)
        for score_col in ("mean_p", "win_rate"):
            combo_rows.append({"group_size": gsize, "variant": vid, "score": score_col,
                               "AUC known vs others": auc(res.loc[m & known, score_col], res.loc[m & other, score_col]),
                               "AUC known vs random": auc(res.loc[m & known, score_col], res.loc[m & rnd_cat, score_col]),
                               "AUC candidate vs random": auc(res.loc[m & cand, score_col], res.loc[m & rnd_cat, score_col])})
combo_tbl = pd.DataFrame(combo_rows).round(3)
display(combo_tbl.sort_values("AUC known vs random", ascending=False))

print("\nカテゴリ別平均（group_size, variant ごと）:")
display(res.groupby(["group_size", "variant", "category"])[["mean_p", "win_rate"]].mean().round(3))

pos_df = pd.DataFrame(diag_positions, columns=["group_size", "position", "prob"])
print("\n位置ごとの平均確率（内容に関係なく偏っていれば、番号自体に釣られている可能性があります）:")
display(pos_df.groupby(["group_size", "position"])["prob"].mean().round(3))

if INCLUDE_NONE and none_diag:
    none_df = pd.DataFrame(none_diag, columns=["group_size", "variant", "prob_none", "is_argmax"])
    print("\n「その他（該当なし）」の選ばれやすさ（group_size, variant ごと）:")
    display(none_df.groupby(["group_size", "variant"]).agg(mean_prob_none=("prob_none", "mean"), argmax_rate=("is_argmax", "mean")).round(3))

## 対話型グラフ（ホバーで遺伝子名）

### このセルがすること：AUCが最も高かった組み合わせを分類別の散布図にし、全組み合わせのAUCをヒートマップで比較する
- 各点にマウスを乗せると記号・タンパク質名・分類・値が出ます。
- ヒートマップは `mean_p` の「known vs random」AUCを、group_size × 質問パターンで並べたものです。
- `pip install plotly nbformat ipython` が必要です（入れた後はカーネルを再起動）。

In [ ]:
import plotly.graph_objects as go

best_row = combo_tbl.sort_values("AUC known vs random", ascending=False).iloc[0]
best_gsize, best_vid, best_score = best_row["group_size"], best_row["variant"], best_row["score"]
print(f"最もAUC known vs randomが高い組み合わせ: group_size={best_gsize}, variant={best_vid}, score={best_score} (AUC={best_row['AUC known vs random']})")

CAT = [("known", "#2a78d6", "circle"), ("candidate", "#eb6834", "square"), ("random", "#1baf7a", "diamond")]
sub = res[(res["group_size"] == best_gsize) & (res["variant"] == best_vid)]
fig = go.Figure()
rng_j = random.Random(0)
for j, (cat, color, sym) in enumerate(CAT):
    d = sub[sub["category"] == cat]
    fig.add_trace(go.Scatter(x=[j + (rng_j.random() - 0.5) * 0.5 for _ in range(len(d))], y=d[best_score],
                             mode="markers", name=cat, marker=dict(color=color, symbol=sym, size=9, opacity=0.8, line=dict(width=1, color="#fcfcfb")),
                             customdata=d[["symbol", "gene_label", "category"]].values,
                             hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<br>値 = %{y:.3f}<extra></extra>"))
fig.update_layout(title=f"{DISEASE}: {best_score} by category (group_size={best_gsize}, variant={best_vid})",
                  xaxis=dict(tickvals=[0, 1, 2], ticktext=[c[0] for c in CAT]), template="plotly_white", width=800, height=480)
fig.show()

heat = combo_tbl[combo_tbl["score"] == "mean_p"].pivot(index="variant", columns="group_size", values="AUC known vs random")
fig2 = go.Figure(data=go.Heatmap(z=heat.values, x=[f"group={c}" for c in heat.columns], y=list(heat.index), colorscale="Blues",
                                 zmin=0.5, zmax=1.0, text=heat.round(3).values, texttemplate="%{text}"))
fig2.update_layout(title=f"{DISEASE}: AUC(known vs random, mean_p) by group size x question variant", template="plotly_white", width=600, height=350)
fig2.show()

html_path = OUT_CSV.replace(".csv", "_charts.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for fg in (fig, fig2):
        f.write(fg.to_html(full_html=False, include_plotlyjs="cdn"))
    f.write("</body></html>")
print("saved:", html_path)